### CRISP-DM Phase 3 - Data Preparation : Laws & Policies

Cleaning and preprocessing of **Climate Change Laws of the World** by *Climate Policy Radar* and **Climate Policy Database** by *NewClimate Institute*

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import ftfy
import sys
import re

In [ ]:
# Load the datasets
cclw = pd.read_csv('data/CCLW-27-04-2026.csv')
cpdb = pd.read_csv('data/CPDB-21-04-2026.csv')

DP1 - Harmonizing

In [ ]:
## Select and rename columns
cclw = cclw[['Family Title', 'Family Summary', 'Geography ISOs', 'First event in timeline', 'Topic/Response', 'Document URL', 'Hazard']].copy()
cclw.columns = ['Title', 'Summary', 'Country', 'Year', 'Topic', 'Link', 'Hazard']
cclw['Source'] = 'CCLW'

cpdb = cpdb[['policy_title', 'policy_description', 'country_iso', 'decision_date', 'policy_objective', 'reference']].copy()
cpdb.columns = ['Title', 'Summary', 'Country', 'Year', 'Topic', 'Link']
cpdb['Hazard'] = None
cpdb['Source'] = 'CPDB'

DP2 - Cleaning

In [ ]:
## Clean HTML format 
def clean_summary(df):    
    def clean(x):
        if pd.isna(x):
            return ''
        x = ftfy.fix_text(str(x))
        x = BeautifulSoup(x, 'html.parser').get_text(separator=' ')
        return x.strip()
    
    df['Summary'] = df['Summary'].apply(clean)
    return df

In [ ]:
## Standardize hazard labels
def map_hazards(df, mapping):
    def map(x, mapping):
        if pd.isna(x):
            return []
        raw_labels = [h.strip().lower() for h in x.split(';')]
        mapped = set()
        for label in raw_labels:
            if label in mapping:
                mapped.add(mapping[label])
        return sorted(list(mapped))
    
    df['Hazard'] = df['Hazard'].apply(lambda x: map(x, mapping))
    return df

In [ ]:
hazard_mapping = {
    # Flood
    'flood':'flood', 'floods':'flood', 'flooding':'flood',
    # Drought 
    'drought':'drought', 'droughts':'drought', 
    # Temperature extremes
    'heat waves and heat stress':'temperature_extremes',
    'heat waves':'temperature_extremes', 'heatwaves':'temperature_extremes',
    'heat wave':'temperature_extremes', 'heat stress':'temperature_extremes',
    'changes in average temperature':'temperature_extremes',
    'change in average temperature':'temperature_extremes', 
    'cold waves':'temperature_extremes', 'cold wave':'temperature_extremes',
    'increases in heat waves':'temperature_extremes', 'heat': 'temperature_extremes',
    # Sea level rise (and coastal hazards)
    'sea level rise':'sea_level_rise', 'coastal erosion':'sea_level_rise',
    'sea level change':'sea_level_rise', 'sea level rise soil erosion':'sea_level_rise',
    # Storm 
    'storms':'storm', 'storm':'storm', 'windstorms':'storm', 'windstorm':'storm',
    'cyclones':'storm', 'hurricanes':'storm', 'hurricane':'storm', 'typhoons':'storm',
    'tropical cyclones':'storm', 'storms, hurricanes, tsunamis, cyclones ':'storm',
    # Wildfire
    'wildfires':'wildfire', 'fires':'wildfire', 'wildfire':'wildfire', 'forest wildfires':'wildfire',
    'wild fire':'wildfire', 'fire':'wildfire', 
    # Snow melt
    'glacial melting':'melting', 'loss of snow cover':'melting', 'snow melt':'melting',
    'loss of ice mass':'melting', 'loss of sea ice':'melting', 'glacial melt':'melting',
    'melting glacial ice':'melting', 'thawing permafrost':'melting', 'permafrost thawing':'melting', 
    # Erosion
    'soil erosion':'erosion', 'erosion':'erosion', 'land erosion':'erosion', 'soil soil erosion':'erosion',
    # Other
    'tsunamis':'other', 'earthquakes':'other', 'earhquakes':'other',  'earthquake':'other', 
    'changes in average precipitation':'other', 'changes in precipitation':'other',
    'changes in soil quality':'other', 'change in soil quality':'other', 'changes in air quality':'other', 
    'change in air quality':'other', 'changes in surface water':'other', 'change in surface water':'other',
    'surface water change':'other', 'changes in groundwater':'other', 'groundwater change':'other',
    'desertification':'other', 'biodiversity loss':'other', 'ocean acidification':'other',  
    'high tides':'other', 'hail/frost':'other', 'hail':'other', 'epidemic':'other', 'diseases':'other', 
    'volcano eruptions':'other', 'forest':'other', 'lightning':'other', 'avalanches':'other', 
    'landslides':'other', 'mudslides':'other', 'landslide':'other', 'glacial':'other', 
    'sea water intrusion':'other'
}

In [ ]:
## Apply cleaning steps to relevant columns
# Clean texts
cclw = clean_summary(cclw)
cpdb = clean_summary(cpdb)

# Map hazards
cclw = map_hazards(cclw, hazard_mapping)
cpdb = map_hazards(cpdb, hazard_mapping)

# Standardize dates
cclw['Year'] = pd.to_datetime(cclw['Year']).dt.year.astype('Int64')
cpdb['Year'] = pd.to_numeric(cpdb['Year']).astype('Int64')

DP3 - Filtering

In [ ]:
print(f"CCLW document count before filtering: {len(cclw)}")
print(f"CPDB document count before filtering: {len(cpdb)}")
print(f"CCLW document count with hazard label before filtering: {len(cclw[cclw['Hazard'].apply(lambda x: len(x) > 0)])}")

# Remove documents with ISO codes 'XAA' or 'XAB' from CCLW
cclw = cclw[~cclw['Country'].isin(['XAA', 'XAB'])].copy()
print(f"CCLW document count with hazard label after first filtering: {len(cclw[cclw['Hazard'].apply(lambda x: len(x) > 0)])}")

# Remove documents with summary shorter than 25 words from CCLW 
cclw = cclw[cclw['Summary'].apply(lambda x: len(x.split()) >= 25)].copy()
print(f"CCLW document count with hazard label after second filtering: {len(cclw[cclw['Hazard'].apply(lambda x: len(x) > 0)])}")

# Remove documents with empty summary from CPDB 
cpdb = cpdb[cpdb['Summary'].apply(lambda x: len(str(x).strip()) > 0)].copy()

print(f"CCLW document count after filtering: {len(cclw)}")
print(f"CPDB document count after filtering: {len(cpdb)}")

DP4 - Merging

In [ ]:
# Remove duplicates of each dataset
cclw = cclw.drop_duplicates(subset=['Title','Summary'], keep='first').copy()
cpdb = cpdb.drop_duplicates(subset=['Title','Summary'], keep='first').copy()

print(f"CCLW document count after removing duplicates: {len(cclw)}")
print(f"CPDB document count after removing duplicates: {len(cpdb)}")
print(f"CCLW document count with hazard label after removing duplicates: {len(cclw[cclw['Hazard'].apply(lambda x: len(x) > 0)])}")

In [ ]:
def strip_brackets_and_short_words(title):
    title = re.sub(r'\(.*?\)', '', str(title))  
    words = [w for w in title.split() if len(w) >= 4]  
    return ' '.join(words).strip().lower()

def has_common_word(title1, title2, min_length=4):
    words1 = set(w.lower() for w in str(title1).split() if len(w) >= min_length)
    words2 = set(w.lower() for w in str(title2).split() if len(w) >= min_length)
    return len(words1 & words2) > 4

In [ ]:
# Remove duplicates between datasets
def interactive_deduplication(cclw, cpdb):
    to_remove = set()

    # Create a temporary unique key to identify rows to remove
    cpdb['temp_key'] = (cpdb['Country'].astype(str) + cpdb['Year'].astype(str) + cpdb['Title'].astype(str))
        
    # Match documents on country and year
    candidates = cpdb.merge(cclw[['Country', 'Year', 'Title']], on=['Country', 'Year'],
        how='inner', suffixes=('_cpdb', '_cclw'))
    
    # Remove exact matches (also by excluding anything in brackets and words with less than 4 chars)
    candidates['temp_key'] = (candidates['Country'].astype(str) + candidates['Year'].astype(str) + candidates['Title_cpdb'].astype(str))

    exact = candidates[
        candidates['Title_cpdb'].apply(strip_brackets_and_short_words) == 
        candidates['Title_cclw'].apply(strip_brackets_and_short_words)]
    to_remove.update(exact['temp_key'].tolist())
    print(f"Duplicates removed: {len(exact)}")  
    
    # Remaining candidates with at least four common words
    non_exact = candidates[~candidates['temp_key'].isin(to_remove)]
    non_exact = non_exact[
        non_exact.apply(
            lambda row: has_common_word(row['Title_cpdb'], row['Title_cclw']), axis=1
        )
    ]
    print(f"Document pairs to review manually: {len(non_exact)}")

    # Review manually
    sys.stdout.flush()
    for i, (_, row) in enumerate(non_exact.iterrows()):
        print(f"Pair {i+1}/{len(non_exact)}")
        print(f"Country: {row['Country']}, Year: {row['Year']}")
        print(f"CPDB: {row['Title_cpdb']}")
        print(f"CCLW: {row['Title_cclw']}")
        print(f"Duplicate? (y/n/q to quit): ", end='')
        sys.stdout.flush()
        
        answer = input().strip().lower()
        if answer == 'q':
            print("Stopped early.")
            break
        elif answer == 'y':
            to_remove.add(row['temp_key'])
        print()
    
    cpdb_clean = cpdb[~cpdb['temp_key'].isin(to_remove)].drop(columns=['temp_key'])
    combined = pd.concat([cclw, cpdb_clean], ignore_index=True)
    return combined

combined = interactive_deduplication(cclw, cpdb)
print(f"Combined document count after removing duplicates: {len(combined)}")

In [ ]:
## Export .csv
# Combined dataset
combined.to_csv('data/law.csv', index=False)

# Benchmark of documents with hazard label
benchmark = combined[combined['Hazard'].apply(lambda x: len(x) > 0)].copy()
benchmark.to_csv('data/benchmark.csv', index=False)